# Verify that the v9 results are saved on Hugging Face (and everything else that is left)

Read-only audit of the model repository. Nothing is trained or written to the Hub. It checks, in order:

1. **Presence**: Part 1 handoff, main store, research store (6 rows), test evaluations, reports, final ZIP and its receipt.
2. **Training**: for every stage (graph and CNN) the recorded stopping reason, epochs, best epoch, protocol label and saved curves.
3. **Evaluation**: each of the 6 rows has all 189 test patients, no duplicates, no missing values, and the same split as Part 1.
4. **Results**: per-row test Dice, hybrid vs CNN-only vs SegResNet with paired bootstrap intervals.
5. **Reports and claims**: report status, Shapley/hop/qualitative outputs, `claims_scope.json`, SLIC pilot.
6. **Safety**: no Hugging Face token inside the saved JSON or the final ZIP.
7. **Part 2 handoff** (new in v4): compares the saved Part 2 state with exactly what Parts 3b/5/6 check, and names any field/type that differs.
8. **Advisor follow-up outputs** (new in v4): `graphsage_hybrid` row and `advisor_followup/*` files; missing = WARN (not run yet).

Each check prints PASS, WARN or FAIL with a reason. A check that cannot run reports ERROR instead of stopping the notebook.
Marimo-safe: no underscore names, every global defined in exactly one cell, loops inside functions.
Token: paste it into `HF_TOKEN_INPUT` (or set the `HF_TOKEN` env var, or use the hidden prompt). Clear it before saving or exporting this notebook.

In [ ]:
import os, io, re, json, hashlib, zipfile, getpass
import numpy as np
import pandas as pd

REPO_ID = "marvelpokemaster/brats-qos-hrgn-model"
REPO_TYPE = "model"
ROWS = ["full", "no_voxel", "cnn_only", "graphsage_backbone", "fixed_shared_hops", "segresnet"]
PILOT_ROWS = ["full", "slic_5000", "slic_10000", "slic_20000"]
SEED = 42
PATIENCE = 15
MAX_EPOCHS = {"graph": 100, "voxel": 100}
EXPECTED_COUNTS = {"train": 875, "val": 187, "test": 189}
EXPECTED_FINGERPRINT = "9e315a86257ac0f68e1f22d6d10bad548d518a08c41ab11f936542939ba2b989"
STAGE1_PREFIX = "stage1_convergence_v9/final"
STAGE2_NAME = "stage2_latest_gpu_v4.pt"
PLAN_KEY = None        # leave None to auto-pick the convergence-v9 store; or set e.g. "6d02e17581d78c4d5c40"
PILOT_KEY = None       # same for the SLIC pilot store, e.g. "fb47bb45a49f43b22f90"
WANT_PROTOCOL = "convergence-v9"
CHECK_CHECKPOINTS = True

# Paste your Hugging Face token between the quotes (leave "" to use HF_TOKEN env var or a hidden prompt).
# Clear it again before saving/exporting this notebook.
HF_TOKEN_INPUT = ""

TOKEN = HF_TOKEN_INPUT.strip() or (os.environ.get("HF_TOKEN") or "").strip() or getpass.getpass("Hugging Face token (hidden): ").strip()

from huggingface_hub import HfApi, HfFileSystem, hf_hub_download
api = HfApi(token=TOKEN)
fs = HfFileSystem(token=TOKEN)
print("ok:", api.whoami()["name"])

In [ ]:
def list_folders(path):
    return [item.path for item in api.list_repo_tree(REPO_ID, repo_type=REPO_TYPE, path_in_repo=path)
            if type(item).__name__ == "RepoFolder"]


def read_catalog(prefix):
    local = hf_hub_download(REPO_ID, prefix + "/catalog.zip", repo_type=REPO_TYPE, token=TOKEN)
    with zipfile.ZipFile(local) as zf:
        return json.loads(zf.read("catalog.json"))


def read_member(catalog, name):
    rec = catalog["files"][name]
    with fs.open("%s/%s" % (REPO_ID, rec["archive"]), "rb") as handle:
        with zipfile.ZipFile(handle) as zf:
            return zf.read(name)


def read_json(catalog, name):
    return json.loads(read_member(catalog, name))


def has(catalog, name):
    return catalog is not None and name in catalog["files"]


def load_torch(blob):
    import torch
    return torch.load(io.BytesIO(blob), map_location="cpu", weights_only=False)


def row(check, ok, detail, level="FAIL"):
    return dict(check=check, status="PASS" if ok else level, detail=str(detail))


def safe(label, fn):
    try:
        return fn()
    except Exception as exc:
        return [dict(check=label, status="ERROR", detail="%s: %s" % (type(exc).__name__, str(exc)[:300]))]


VAL_KEYS = ("val_dice", "val_mean_dice", "val_dice_mean", "validation_dice")
TRAIN_LOSS = ("train_loss", "loss", "train_objective")
VAL_LOSS = ("val_loss", "val_total", "validation_loss")
TRAIN_DICE = ("train_patch_dice", "train_node_proxy", "train_dice", "train_mean_dice")


def pick(hist, candidates):
    for cand in candidates:
        if hist and cand in hist[0]:
            return cand
    return None


def recompute(hist, max_epochs, patience):
    key = pick(hist, VAL_KEYS)
    if not hist or key is None:
        return dict(verdict="NO_HISTORY", epochs=len(hist or []), best_epoch=None, stale=None, best_val=None)
    scores = np.array([entry[key] for entry in hist], dtype=float)
    best_i = int(np.argmax(scores))
    stale = len(scores) - 1 - best_i
    if stale >= patience:
        label = "EARLY_STOP"
    elif len(scores) >= max_epochs:
        label = "EPOCH_CAP"
    else:
        label = "TIME_CAP"
    return dict(verdict=label, epochs=len(scores), best_epoch=int(hist[best_i].get("epoch", best_i + 1)),
                stale=int(stale), best_val=float(scores[best_i]))

In [ ]:
NEED_TRAINED = {"runs/%s/seed_%d/trained.json" % (r, SEED) for r in ROWS}
PILOT_TRAINED = {"runs/%s/seed_%d/trained.json" % (r, SEED) for r in PILOT_ROWS}


def discover_stores():
    found = {}
    for top in list_folders("continuation_v8"):
        for rdir in list_folders(top + "/research"):
            try:
                cat = read_catalog(rdir)
            except Exception as exc:
                print("skip", rdir, "->", type(exc).__name__)
                continue
            names = set(cat["files"])
            try:
                manifest = json.loads(read_member(cat, "protocol.json"))
                label = (manifest.get("compute_protocol") or {}).get("protocol", "practical-v8")
            except Exception:
                label = "unknown"
            found[rdir] = dict(catalog=cat, main=len(NEED_TRAINED & names), pilot=len(PILOT_TRAINED & names),
                               tests=sum(n.startswith("evaluation/test/") for n in names), protocol=label)
    return found


def choose(stores, kind):
    key = PLAN_KEY if kind == "main" else PILOT_KEY
    if key:
        return [k for k in stores if k.endswith("/" + key)][:1]
    if kind == "main":
        good = [k for k, v in stores.items() if v["main"] == len(NEED_TRAINED) and v["protocol"] == WANT_PROTOCOL]
        return sorted(good, key=lambda k: -stores[k]["tests"])[:1]
    good = [k for k, v in stores.items() if v["pilot"] >= 3 and v["main"] < len(NEED_TRAINED) and v["protocol"] == WANT_PROTOCOL]
    return sorted(good, key=lambda k: -stores[k]["pilot"])[:1]


STORES = discover_stores()
for store_name, info in sorted(STORES.items()):
    print("  %-72s main %d/6  pilot %d/4  tests %d  %s" % (store_name, info["main"], info["pilot"], info["tests"], info["protocol"]))
MAIN_PICK = choose(STORES, "main")
PILOT_PICK = choose(STORES, "pilot")
STORE = MAIN_PICK[0] if MAIN_PICK else None
PILOT_STORE = PILOT_PICK[0] if PILOT_PICK else None
CATALOG = STORES[STORE]["catalog"] if STORE else None
PILOT_CATALOG = STORES[PILOT_STORE]["catalog"] if PILOT_STORE else None
ROOT_PREFIX = STORE.split("/research/")[0] if STORE else None


def try_catalog(prefix):
    try:
        return read_catalog(prefix)
    except Exception as exc:
        print("could not read", prefix, "->", type(exc).__name__)
        return None


ADV_CATALOG = try_catalog(ROOT_PREFIX + "/advisor_reports") if ROOT_PREFIX else None
MAIN_CATALOG = try_catalog(ROOT_PREFIX + "/main") if ROOT_PREFIX else None
print("main research store:", STORE)
print("pilot store:", PILOT_STORE)
print("advisor_reports store:", None if ADV_CATALOG is None else ROOT_PREFIX + "/advisor_reports", "| main store:", None if MAIN_CATALOG is None else ROOT_PREFIX + "/main")

In [ ]:
def presence_rows():
    out = []
    out.append(row("Research store found with all 6 trained rows", STORE is not None, STORE or "none found; set PLAN_KEY"))
    if STORE is None:
        return out
    out.append(row("store was trained under " + WANT_PROTOCOL + " (older stores are time-capped and must not be used)",
                   STORES[STORE]["protocol"] == WANT_PROTOCOL, "%s protocol=%s" % (STORE, STORES[STORE]["protocol"])))
    out.append(row("Stage-1 sha prefix in path", len(ROOT_PREFIX.split("/")[-1]) == 20, ROOT_PREFIX))
    names = set(CATALOG["files"])
    for r in ROWS:
        out.append(row("trained.json: %s" % r, "runs/%s/seed_%d/trained.json" % (r, SEED) in names, ""))
    for r in ROWS:
        out.append(row("test evaluation: %s" % r, "evaluation/test/%s/seed_%d.json" % (r, SEED) in names, ""))
    for name in ("protocol.json", "session_status.json", "reports/test_report.json", "reports/test_summary.csv",
                 "reports/test_patients.csv", "reports/qualitative/selection.json"):
        out.append(row("research store file: " + name, name in names, "", level="WARN" if name.startswith("reports/") else "FAIL"))
    curves = [n for n in names if n.startswith("reports/curves/")]
    out.append(row("training-curve images", len(curves) >= 1, "%d found" % len(curves), level="WARN"))
    quals = [n for n in names if n.startswith("reports/qualitative/") and n.endswith(".png")]
    out.append(row("qualitative examples (advisor asks 5 to 10)", 5 <= len(quals) <= 10, "%d images" % len(quals), level="WARN"))
    out.append(row("advisor_reports store exists", ADV_CATALOG is not None, ROOT_PREFIX + "/advisor_reports"))
    adv = set(ADV_CATALOG["files"]) if ADV_CATALOG is not None else set()
    for name in ("report_status.json", "final_reports.zip", "final_export_receipt.json", "reports/claims_scope.json",
                 "reports/scope.json", "reports/main_training_history.json", "reports/slic_pilot_scope.json",
                 "reports/region_shapley_summary.json", "reports/hop_summary.json", "reports/region_shapley.png",
                 "reports/hop_distributions.png", "reports/local_graph.png"):
        out.append(row("advisor_reports file: " + name, name in adv, "", level="FAIL" if name in ("report_status.json", "final_reports.zip") else "WARN"))
    xai = [n for n in adv if n.startswith("reports/xai/")]
    out.append(row("Shapley cases (expected 10)", len(xai) == 10, "%d" % len(xai), level="WARN"))
    nodes = [n for n in adv if n.startswith("reports/nodes/")]
    out.append(row("hop/gate node diagnostics (one per test patient)", len(nodes) == EXPECTED_COUNTS["test"], "%d" % len(nodes), level="WARN"))
    out.append(row("main store catalog", True, "checked below"))
    try:
        main_cat = read_catalog(ROOT_PREFIX + "/main")
        out.append(row("main store holds " + STAGE2_NAME, STAGE2_NAME in main_cat["files"], ROOT_PREFIX + "/main"))
    except Exception as exc:
        out.append(row("main store catalog", False, type(exc).__name__ + ": " + str(exc)[:200]))
    try:
        stage1_cat = read_catalog(STAGE1_PREFIX)
        out.append(row("Part 1 handoff " + STAGE1_PREFIX, True, ", ".join(sorted(stage1_cat["files"]))[:200]))
    except Exception as exc:
        out.append(row("Part 1 handoff " + STAGE1_PREFIX, False, type(exc).__name__ + ": " + str(exc)[:200]))
    return out


PRESENCE = safe("presence", presence_rows)

In [ ]:
def read_stage1():
    stage1_cat = read_catalog(STAGE1_PREFIX)
    ckpt_name = [n for n in stage1_cat["files"] if n.endswith("stage1_graph_model.pt")]
    if not ckpt_name:
        raise RuntimeError("no stage1_graph_model.pt in the handoff catalog: " + ", ".join(stage1_cat["files"]))
    blob = read_member(stage1_cat, ckpt_name[0])
    return load_torch(blob), hashlib.sha256(blob).hexdigest()


def stage1_rows():
    if not CHECK_CHECKPOINTS:
        return [row("Part 1 checkpoint", True, "skipped (CHECK_CHECKPOINTS False)")]
    ckpt, digest = read_stage1()
    out = []
    split = ckpt.get("split") or {}
    counts = {k: len(v) for k, v in split.items()}
    out.append(row("split counts 875/187/189", counts == EXPECTED_COUNTS, counts))
    fingerprint = hashlib.sha256(json.dumps(split, sort_keys=True).encode()).hexdigest()
    out.append(row("split fingerprint unchanged", fingerprint == EXPECTED_FINGERPRINT, fingerprint[:16] + "..."))
    out.append(row("stage-1 sha256 prefix matches store path", ROOT_PREFIX is not None and ROOT_PREFIX.endswith(digest[:20]),
                   "checkpoint sha " + digest[:20], level="WARN"))
    hist = ckpt.get("history") or []
    res = recompute(hist, MAX_EPOCHS["graph"], PATIENCE)
    out.append(row("Part 1 graph stage ended by early stopping or epoch cap", res["verdict"] in ("EARLY_STOP", "EPOCH_CAP"),
                   "recorded=%s recomputed=%s epochs=%s best=%s" % (ckpt.get("stopping_reason"), res["verdict"], res["epochs"], res["best_epoch"])))
    out.append(row("Part 1 protocol label is convergence-v9", ckpt.get("compute_protocol") == "convergence-v9",
                   "label=%s (a 'practical-v8' label is a stale provenance string, not a training problem)" % ckpt.get("compute_protocol"), level="WARN"))
    out.append(row("Part 1 curves saved (train/val loss and Dice)", all(pick(hist, c) for c in (TRAIN_LOSS, VAL_LOSS, VAL_KEYS)),
                   "keys=%s" % (list(hist[0]) if hist else None), level="WARN"))
    return out


STAGE1_ROWS = safe("stage-1 checkpoint", stage1_rows)

In [ ]:
def trained_table():
    records = []
    for r in ROWS:
        rec = read_json(CATALOG, "runs/%s/seed_%d/trained.json" % (r, SEED))
        for stage, proto in rec["protocols"].items():
            hist = proto.get("history") or []
            res = recompute(hist, MAX_EPOCHS[stage], PATIENCE)
            records.append(dict(row=r, stage=stage, recorded=proto.get("stopping_reason"), recomputed=res["verdict"],
                                epochs=res["epochs"], best_epoch=res["best_epoch"], stale=res["stale"], best_val=res["best_val"],
                                convergence=proto.get("convergence_demonstrated"), protocol=proto.get("protocol"),
                                sessions=proto.get("sessions"), has_curves=bool(hist) and all(pick(hist, c) for c in (TRAIN_LOSS, VAL_LOSS)),
                                minutes=(proto.get("training_seconds") or proto.get("phase_seconds_used") or 0) / 60))
    return pd.DataFrame(records)


def training_rows(table):
    out = []
    own = table[table.recorded != "imported_selection"]
    imported = table[table.recorded == "imported_selection"]
    for rec in own.to_dict("records"):
        ok = rec["recorded"] in ("early_stopping", "epoch_cap") and rec["recomputed"] in ("EARLY_STOP", "EPOCH_CAP")
        out.append(row("stopping: %s/%s" % (rec["row"], rec["stage"]), ok,
                       "recorded=%s recomputed=%s epochs=%s best=%s since-best=%s sessions=%s" % (
                           rec["recorded"], rec["recomputed"], rec["epochs"], rec["best_epoch"], rec["stale"], rec["sessions"])))
        out.append(row("curves saved: %s/%s" % (rec["row"], rec["stage"]), rec["has_curves"], "", level="WARN"))
        if rec["recomputed"] == "EPOCH_CAP":
            out.append(row("convergence demonstrated: %s/%s" % (rec["row"], rec["stage"]), False,
                           "ran the full %d epochs; best epoch %s, %s epochs since best. Report it as epoch_cap, not as converged, and say validation was still improving" % (
                               rec["epochs"], rec["best_epoch"], rec["stale"]), level="WARN"))
    for rec in imported.to_dict("records"):
        out.append(row("imported from Part 1/2: %s/%s" % (rec["row"], rec["stage"]), True, "checked in the Part 1 and Part 2 checks"))
    return out


TRAINED_TABLE = safe("trained.json", trained_table)
if isinstance(TRAINED_TABLE, pd.DataFrame):
    pd.set_option("display.width", 220)
    print(TRAINED_TABLE.to_string(index=False))
    TRAINING = safe("training", lambda: training_rows(TRAINED_TABLE))
else:
    TRAINING = TRAINED_TABLE

In [ ]:
def stage2_rows():
    if not CHECK_CHECKPOINTS:
        return [row("Part 2 CNN stage", True, "skipped (CHECK_CHECKPOINTS False)")]
    main_cat = read_catalog(ROOT_PREFIX + "/main")
    state = load_torch(read_member(main_cat, STAGE2_NAME))
    hist = state.get("vox_history") or []
    res = recompute(hist, MAX_EPOCHS["voxel"], PATIENCE)
    out = [row("Part 2 hybrid CNN stage ended by early stopping or epoch cap", res["verdict"] in ("EARLY_STOP", "EPOCH_CAP"),
               "recomputed=%s epochs=%s best epoch=%s best val Dice=%s since-best=%s" % (
                   res["verdict"], res["epochs"], res["best_epoch"], None if res["best_val"] is None else round(res["best_val"], 4), res["stale"]))]
    out.append(row("Part 2 curves saved", all(pick(hist, c) for c in (TRAIN_LOSS, VAL_LOSS, VAL_KEYS)),
                   "keys=%s" % (list(hist[0]) if hist else None), level="WARN"))
    return out


STAGE2_ROWS = safe("stage-2 checkpoint", stage2_rows)

In [ ]:
def metric_groups(metrics):
    groups = {}
    for key, value in metrics.items():
        low = key.lower()
        if "dice" not in low or "raw" in low:
            continue
        for region in ("wt", "tc", "et"):
            if re.search(r"(^|[^a-z])%s([^a-z]|$)" % region, low):
                groups[region] = key
    return groups


def load_test(row_name):
    return read_json(CATALOG, "evaluation/test/%s/seed_%d.json" % (row_name, SEED))


def evaluation_rows():
    out = []
    expected_ids = None
    if CHECK_CHECKPOINTS:
        ckpt = read_stage1()[0]
        expected_ids = set(ckpt["split"]["test"])
    for r in ROWS:
        res = load_test(r)
        patients = res.get("patients", [])
        ids = [p["case_id"] for p in patients]
        out.append(row("test patients: %s" % r, len(patients) == EXPECTED_COUNTS["test"] and len(set(ids)) == len(ids),
                       "%d patients, %d unique, complete=%s" % (len(patients), len(set(ids)), res.get("complete"))))
        if expected_ids is not None:
            out.append(row("same test set as Part 1 split: %s" % r, set(ids) == expected_ids, "missing %d, unexpected %d" % (
                len(expected_ids - set(ids)), len(set(ids) - expected_ids))))
        bad = 0
        for p in patients:
            for value in p["metrics"].values():
                if isinstance(value, (int, float)) and not np.isfinite(value):
                    bad += 1
        out.append(row("finite metrics: %s" % r, bad == 0, "%d non-finite values (HD95 of an empty region may be recorded as inf/NaN by convention)" % bad, level="WARN"))
    return out


EVALUATION = safe("evaluation", evaluation_rows)

In [ ]:
def region_dice(patients):
    first = patients[0]["metrics"]
    groups = metric_groups(first)
    if set(groups) != {"wt", "tc", "et"}:
        raise RuntimeError("could not find WT/TC/ET Dice keys in %s" % list(first))
    arr = np.array([[p["metrics"][groups[g]] for g in ("wt", "tc", "et")] for p in patients], dtype=float)
    return [p["case_id"] for p in patients], arr


def paired(a_ids, a_arr, b_ids, b_arr, draws=5000):
    common = sorted(set(a_ids) & set(b_ids))
    ia = {c: i for i, c in enumerate(a_ids)}
    ib = {c: i for i, c in enumerate(b_ids)}
    diff = np.array([a_arr[ia[c]].mean() - b_arr[ib[c]].mean() for c in common])
    rng = np.random.default_rng(0)
    boot = rng.choice(diff, size=(draws, len(diff)), replace=True).mean(axis=1)
    return float(diff.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)), len(common)


def results_summary():
    data = {r: region_dice(load_test(r)["patients"]) for r in ROWS}
    table = []
    for r in ROWS:
        arr = data[r][1]
        table.append(dict(row=r, WT=arr[:, 0].mean(), TC=arr[:, 1].mean(), ET=arr[:, 2].mean(), mean=arr.mean(),
                          sd_of_patient_mean=arr.mean(axis=1).std(ddof=1)))
    comparisons = []
    for a, b in (("full", "cnn_only"), ("full", "no_voxel"), ("full", "segresnet"), ("cnn_only", "segresnet"),
                 ("no_voxel", "graphsage_backbone"), ("no_voxel", "fixed_shared_hops")):
        mean, lo, hi, n = paired(data[a][0], data[a][1], data[b][0], data[b][1])
        comparisons.append(dict(comparison="%s - %s" % (a, b), mean_dice_diff=mean, ci95_low=lo, ci95_high=hi, patients=n,
                                reading="positive and CI above 0" if lo > 0 else ("negative and CI below 0" if hi < 0 else "CI includes 0")))
    return pd.DataFrame(table), pd.DataFrame(comparisons)


RESULTS = safe("results", results_summary)
if isinstance(RESULTS, tuple):
    RESULTS_TABLE, COMPARISONS = RESULTS
    pd.set_option("display.float_format", lambda v: "%.4f" % v)
    print("TEST Dice per row (raw patient means; see the report JSON for post-processed numbers)")
    print(RESULTS_TABLE.to_string(index=False))
    print()
    print("Paired comparisons (patient-level bootstrap, 95% CI)")
    print(COMPARISONS.to_string(index=False))
    full_vs_seg = COMPARISONS[COMPARISONS.comparison == "full - segresnet"].iloc[0]
    RESULT_ROWS = [row("hybrid is at least as good as SegResNet", full_vs_seg.ci95_high >= 0 and full_vs_seg.mean_dice_diff >= 0,
                       "difference %.4f (CI %.4f to %.4f). If negative, the paper cannot claim the hybrid beats a plain voxel baseline." % (
                           full_vs_seg.mean_dice_diff, full_vs_seg.ci95_low, full_vs_seg.ci95_high), level="WARN")]
    full_vs_cnn = COMPARISONS[COMPARISONS.comparison == "full - cnn_only"].iloc[0]
    RESULT_ROWS.append(row("graph adds to the CNN (hybrid vs CNN-only)", full_vs_cnn.ci95_low > 0,
                           "difference %.4f (CI %.4f to %.4f)" % (full_vs_cnn.mean_dice_diff, full_vs_cnn.ci95_low, full_vs_cnn.ci95_high), level="WARN"))
else:
    RESULT_ROWS = RESULTS

In [ ]:
def report_rows():
    out = []
    status = read_json(ADV_CATALOG, "report_status.json")
    out.append(row("report_status complete", status.get("complete") is True and not status.get("paused"),
                   {k: status.get(k) for k in ("complete", "paused", "advisor_evidence_complete", "publication_ready_certified")}))
    out.append(row("advisor_evidence_complete", status.get("advisor_evidence_complete") is True, ""))
    out.append(row("publication_ready_certified (expected False: the package never certifies itself)", status.get("publication_ready_certified") is False, ""))
    claims = read_json(ADV_CATALOG, "reports/claims_scope.json")
    out.append(row("training_is_compute_limited is False", claims.get("training_is_compute_limited") is False,
                   claims.get("stopping_reasons")))
    out.append(row("single seed disclosed", claims.get("single_seed") is True, ""))
    receipt = read_json(ADV_CATALOG, "final_export_receipt.json")
    direct = "%s/final_reports.zip" % ROOT_PREFIX
    local = hf_hub_download(REPO_ID, direct, repo_type=REPO_TYPE, token=TOKEN)
    h = hashlib.sha256()
    with open(local, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            h.update(block)
    out.append(row("final_reports.zip on the Hub matches its receipt", receipt.get("sha256") == h.hexdigest(),
                   "%s (%d bytes)" % (direct, os.path.getsize(local))))
    with zipfile.ZipFile(local) as zf:
        members = zf.namelist()
        out.append(row("final ZIP contents", len(members) > 10, "%d files: %s" % (len(members), ", ".join(members[:12]) + " ...")))
        leaks = 0
        for name in members:
            if name.endswith((".json", ".csv", ".txt", ".md")) and zf.getinfo(name).file_size < 20_000_000:
                leaks += len(re.findall(r"hf_[A-Za-z0-9]{25,}", zf.read(name).decode("utf-8", "ignore")))
    out.append(row("no Hugging Face token inside the final ZIP", leaks == 0, "%d matches" % leaks))
    return out


def pilot_rows():
    if PILOT_CATALOG is None:
        return [row("SLIC pilot store", False, "not found", level="WARN")]
    out = []
    names = set(PILOT_CATALOG["files"])
    for r in PILOT_ROWS:
        out.append(row("pilot trained: %s" % r, "runs/%s/seed_%d/trained.json" % (r, SEED) in names, ""))
    receipt = None
    where = None
    for label, cat, member in (("pilot store", PILOT_CATALOG, "slic_pilot_receipt.json"), ("main store", MAIN_CATALOG, "slic_pilot_receipt.json"),
                               ("advisor_reports", ADV_CATALOG, "reports/slic_pilot_scope.json")):
        if receipt is None and has(cat, member):
            receipt, where = read_json(cat, member), label
    out.append(row("pilot receipt complete (validation-only, not conclusive)", receipt is not None and receipt.get("complete") is True,
                   "not found in the pilot, main or advisor_reports store" if receipt is None else dict({k: receipt.get(k) for k in ("plan", "scope", "complete", "conclusive_resolution_selection")}, found_in=where), level="WARN"))
    return out


REPORTS = safe("reports", report_rows)
PILOT = safe("pilot", pilot_rows)

In [ ]:
def secret_rows():
    leaks = []
    for label, cat in (("research", CATALOG), ("pilot", PILOT_CATALOG), ("advisor", ADV_CATALOG), ("main", MAIN_CATALOG)):
        if cat is None:
            continue
        for name in cat["files"]:
            if name.endswith(".json") and not name.startswith("runs/"):
                try:
                    text = read_member(cat, name).decode("utf-8", "ignore")
                except Exception:
                    continue
                if re.search(r"hf_[A-Za-z0-9]{25,}", text):
                    leaks.append(label + ":" + name)
    return [row("no Hugging Face token inside saved JSON records", not leaks, leaks or "clean")]


SECRETS = safe("secrets", secret_rows)

In [ ]:
# ---- Part 2 handoff compatibility: exactly what the Part 5 / Part 3b / Part 6 notebooks compare ----
# Their cell raises "Part 2 handoff mismatch: <field>" when the saved Part 2 state differs from the current run.
EXPECTED_STAGE2_CONFIG = {
    "phase_seconds_limit": None, "compute_protocol": "convergence-v9",
    "gpu_training_protocol": "v4-patient-mean-ce", "effective_batch_size": 16,
    "precision_policy": "auto",
    "voxel_epochs": 100, "voxel_lr": 1e-3,
    "voxel_base_channels": 32, "voxel_dice_weight": 1.0,
    "voxel_focal_weight": 0.0, "voxel_focal_gamma": 2.0,
    "ce_weight": 1.0,
    "voxel_boundary_weight": 0.0, "voxel_margin": 10,
    "voxel_max_size": 128, "inference_overlap": 0.25,
    "training_patch": "50pct-tumour-50pct-uniform", "selection": "full-volume-pp-Dice",
}


def canon(x):
    if isinstance(x, dict):
        return {str(k): canon(v) for k, v in sorted(x.items(), key=lambda kv: str(kv[0]))}
    if isinstance(x, (list, tuple)):
        return [canon(v) for v in x]
    if hasattr(x, "item") and getattr(x, "shape", None) == ():
        return x.item()
    return x


def short(x, n=110):
    return repr(x)[:n]


def handoff_rows():
    ckpt, digest = read_stage1()
    main_cat = read_catalog(ROOT_PREFIX + "/main")
    state = load_torch(read_member(main_cat, STAGE2_NAME))
    out = [row("Part 2 state says training_complete", state.get("training_complete") is True, "keys: " + ", ".join(sorted(state))[:180])]
    expected = {"stage1_sha256": digest, "split": ckpt["split"], "dataset_fingerprint": ckpt["dataset_fingerprint"],
                "stage2_config": EXPECTED_STAGE2_CONFIG}
    for key, want in expected.items():
        have = state.get(key)
        if have == want:
            out.append(row("handoff field matches exactly: " + key, True, "type %s" % type(have).__name__))
        elif canon(have) == canon(want):
            out.append(row("handoff field matches exactly: " + key, False,
                           "same content but different nested types (tuple vs list, or numpy vs Python scalar). The revised notebooks accept this.", level="WARN"))
        else:
            detail = "saved %s vs expected %s" % (type(have).__name__, type(want).__name__)
            if isinstance(have, dict) and isinstance(want, dict):
                a, b = canon(have), canon(want)
                diff = sorted(k for k in set(a) | set(b) if a.get(k, "<absent>") != b.get(k, "<absent>"))
                detail += "; differing entries: " + "; ".join("%s saved=%s expected=%s" % (k, short(a.get(k, "<absent>")), short(b.get(k, "<absent>"))) for k in diff[:10])
            out.append(row("handoff field matches exactly: " + key, False, detail))
    return out


HANDOFF = safe("part 2 handoff", handoff_rows)
for item in HANDOFF:
    print(item["status"], "|", item["check"], "|", item["detail"][:260])

In [ ]:
# ---- Advisor follow-up outputs (Part 3b and Part 6). Missing files are WARN = "not run yet", not a failure. ----
ADV_FILES = ["advisor_followup/hgt_vs_sage_with_cnn.json", "advisor_followup/cnn_error_correction.json",
             "advisor_followup/edge_perturbation.json", "advisor_followup/modality_intervention.json",
             "advisor_followup/hop_sensitivity.json", "advisor_followup/et_corrections.png", "advisor_followup/index.json"]
HYB_FILES = ["runs/graphsage_hybrid/seed_%d/trained.json" % SEED, "evaluation/test/graphsage_hybrid/seed_%d.json" % SEED]


def advisor_followup_rows():
    names = set(CATALOG["files"])
    out = []
    for name in HYB_FILES + ADV_FILES:
        out.append(row("follow-up output: " + name, name in names, "present" if name in names else "not on the Hub yet (run Part 3b / Part 6)", level="WARN"))
    if HYB_FILES[1] in names:
        res = read_json(CATALOG, HYB_FILES[1])
        ids = [p["case_id"] for p in res.get("patients", [])]
        out.append(row("graphsage_hybrid: 189 unique test patients, complete", len(ids) == EXPECTED_COUNTS["test"] and len(set(ids)) == len(ids) and res.get("complete") is True,
                       "%d patients, complete=%s" % (len(ids), res.get("complete"))))
        trained = read_json(CATALOG, HYB_FILES[0]) if HYB_FILES[0] in names else {}
        for stage, proto in (trained.get("protocols") or {}).items():
            rec = recompute(proto.get("history") or [], MAX_EPOCHS[stage], PATIENCE)
            out.append(row("graphsage_hybrid/%s stopping" % stage, rec["verdict"] in ("EARLY_STOP", "EPOCH_CAP"),
                           "recorded=%s recomputed=%s epochs=%s best=%s" % (proto.get("stopping_reason"), rec["verdict"], rec["epochs"], rec["best_epoch"])))
    if ADV_FILES[0] in names:
        cmp_ = read_json(CATALOG, ADV_FILES[0])
        print("HGT+CNN (full) vs GraphSAGE+CNN, same 3D CNN, test n=%s" % cmp_.get("n_patients"))
        for model, vals in cmp_["summary"].items():
            print("  %-20s Dice WT/TC/ET/mean = %.2f / %.2f / %.2f / %.2f" % (model, 100 * vals["Dice_WT"], 100 * vals["Dice_TC"], 100 * vals["Dice_ET"], 100 * vals["Dice_mean"]))
        diff = cmp_["comparisons"].get("full - graphsage_hybrid", {}).get("Dice_mean")
        if diff:
            print("  full - graphsage_hybrid mean Dice: %+.2f [%+.2f, %+.2f] holm p=%.3f" % (100 * diff["mean_diff"], 100 * diff["lo"], 100 * diff["hi"], diff["holm_p"]))
    return out


ADVISOR = safe("advisor follow-up", advisor_followup_rows)

In [ ]:
ALL_ROWS = []
for block in (PRESENCE, STAGE1_ROWS, STAGE2_ROWS, TRAINING, EVALUATION, RESULT_ROWS, REPORTS, PILOT, SECRETS, HANDOFF, ADVISOR):
    ALL_ROWS.extend(block)
SUMMARY = pd.DataFrame(ALL_ROWS)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 250)
print(SUMMARY[SUMMARY.status != "PASS"].to_string(index=False) if (SUMMARY.status != "PASS").any() else "Every check passed.")
print()
print(SUMMARY.status.value_counts().to_string())
print()
if (SUMMARY.status == "FAIL").any() or (SUMMARY.status == "ERROR").any():
    print("RESULT: NOT READY. Fix every FAIL/ERROR above (a FAIL on a stopping reason means retrain that stage).")
elif (SUMMARY.status == "WARN").any():
    print("RESULT: data is saved and complete; read the WARN lines before writing claims.")
else:
    print("RESULT: everything verified.")
SUMMARY.to_csv("hub_verification_summary.csv", index=False)
print("saved hub_verification_summary.csv (no token inside).")